# 06 — Blocking

**Blocking is about reducing computational scale.** Without it, matcher compares every left row to every right row—that's N×M pairs (or N² in deduplication). On small data that's fine; on large data it blows up. **Blocking** restricts comparisons to rows that share the same value of a **blocking key** (e.g. `zip_code` or first letter of last name, `last_initial`): only rows in the same block are compared, so total comparisons drop from N×M to the sum of (block_left × block_right) over blocks. The tradeoff: if a true pair has *different* key values, they're in different blocks and never compared, so recall can drop. Choose a key that true pairs are likely to share, or run without blocking when no key is safe. **Nulls:** today, rows where the blocking key is null are grouped into one block (compared only to each other); see below.

**Back:** [05 Design Your Algorithm](05_design_algorithm.ipynb) · **Next:** [07 Deduplication](07_deduplication.ipynb)

## 1. Load data and ground truth

Same data as 02–05: **entity resolution** (500 left, 500 right, 30 true pairs). Load raw, **standardize** (aligned schema, `email_clean`, `full_name`), add **first letter of last name** (`last_initial`) as a blocking key we'll use later, then build the matcher. Using 500×500 makes the scale message clear: without blocking that's 250,000 comparisons; with blocking (e.g. on `zip_code` or `last_initial`) it drops to the sum of within-block pairs. Many true pairs have different zips left vs right, so blocking on zip will miss them (recall drops)—you'll see both the comparison reduction and the tradeoff.

In [1]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_entity_resolution, standardize_for_matching

left_raw, right_raw, ground_truth = load_entity_resolution()
left, right = standardize_for_matching(left_raw, right_raw)
# First letter of last name — useful as a blocking key for the name rule
last_initial = pl.col("last_name").cast(pl.Utf8).str.slice(0, 1).alias("last_initial")
left = left.with_columns(last_initial)
right = right.with_columns(last_initial)
matcher = Matcher(left=left, right=right, left_id="id", right_id="id")
print(f"Left: {left.shape[0]} rows, Right: {right.shape[0]} rows, Ground truth: {ground_truth.shape[0]} pairs")

Left: 500 rows, Right: 500 rows, Ground truth: 30 pairs


## 2. Comparison count: without vs with blocking

Without blocking, matcher does **every left × every right** = N×M comparisons. With blocking, it only compares rows that share the same blocking key value (e.g. `zip_code` or first letter of last name, `last_initial`); each (left_block, right_block) contributes block_size_left × block_size_right. Below we use `zip_code`; you can use any column the same way. (Blocks are formed from key values that appear in **both** left and right; key values that appear on only one side never get compared.)

In [2]:
key = "last_initial"
n_left, n_right = left.height, right.height
no_block_comparisons = n_left * n_right

# With blocking: blocks = key values in BOTH tables; nulls = one block if both sides have nulls
left_counts = left.group_by(key).len()
right_counts = right.group_by(key).len()
common = left_counts.join(right_counts, on=key, how="inner")
with_block_comparisons = (common["len"] * common["len_right"]).sum()

print(f"Without blocking: {n_left} × {n_right} = {no_block_comparisons:,} comparisons")
print(f"With blocking ({key}):  {with_block_comparisons:,} comparisons")
if with_block_comparisons > 0:
    print(f"Reduction: {no_block_comparisons / with_block_comparisons:.1f}× fewer")

Without blocking: 500 × 500 = 250,000 comparisons
With blocking (last_initial):  14,529 comparisons
Reduction: 17.2× fewer


## 3. How nulls are treated (today)

Rows where the **blocking key is null** are not dropped. In matcher today: if **both** left and right have at least one null in the blocking key, all left rows with null and all right rows with null are grouped into a single block and compared only to each other. So null is treated as one block value. Key values that appear on only one side (or only on left or only on right) do not form a block—those rows are never compared for that key.

## 4. With vs without blocking

Run the **same rules as 02–05**: match on `email_clean` or on exact `first_name` + `last_name`. Compare with and without a blocking key (e.g. `zip_code` or `last_initial`). Evaluate on the same ground truth. If recall is lower with blocking, your key is splitting true pairs—those pairs are in different blocks and never compared.

In [3]:
rules = ["email_clean", ["first_name", "last_name"]]  # same as 05
results_no_block = matcher.match(rules=rules)
results_with_block = matcher.match(rules=rules, blocking_key="zip_code")

m_no = results_no_block.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
m_block = results_with_block.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")

print("No blocking:    ", f"recall={m_no['recall']:.2%}, matches={results_no_block.count}")
print("With zip_code: ", f"recall={m_block['recall']:.2%}, matches={results_with_block.count}")

No blocking:     recall=66.67%, matches=20
With zip_code:  recall=33.33%, matches=10


On this data, many true pairs have *different* zips on left vs right, so blocking by `zip_code` puts them in different blocks and recall drops. So: blocking cuts comparisons (scale) but can miss pairs when the key splits them. Choose a key that true pairs are likely to share, or run without blocking when no key is safe.

### 4a. Timing: with vs without blocking (`last_initial`)

To show blocking speedup, we **synthetically 50× the data** for this example only: replicate left, right, and ground truth 50 times with unique IDs (25,000×25,000 = 625M comparisons without blocking). We build a **fresh matcher** on that and time with and without `last_initial` blocking—first **exact** rules, then **fuzzy**. Fuzzy matching is costlier per comparison (string similarity), so the speedup from blocking is often **more pronounced** with fuzzy than with exact. **For small data the overhead may not be worth it**; here we use large data so the benefit is clear.

In [7]:
import time

N_COPIES = 50 # 500 → 50k rows each; 2.5B pairs without blocking
left_big = pl.concat([left.with_columns((pl.col("id") + f"_c{i}").alias("id")) for i in range(N_COPIES)])
right_big = pl.concat([right.with_columns((pl.col("id") + f"_c{i}").alias("id")) for i in range(N_COPIES)])
gt_big = pl.concat([
    ground_truth.with_columns(
        (pl.col("left_id") + f"_c{i}").alias("id"),
        (pl.col("right_id") + f"_c{i}").alias("id_right"),
    ).select(["id", "id_right"])
    for i in range(N_COPIES)
])
last_initial = pl.col("last_name").cast(pl.Utf8).str.slice(0, 1).alias("last_initial")
left_big = left_big.with_columns(last_initial)
right_big = right_big.with_columns(last_initial)
matcher_big = Matcher(left=left_big, right=right_big, left_id="id", right_id="id")

rules = [["full_name"]]
N_TRIALS = 3
print(f"Timing on {left_big.height:,}×{right_big.height:,} (synthetic {N_COPIES}×), mean of {N_TRIALS} trials...")

times_no = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    matcher_big.match(rules=rules)
    times_no.append(time.perf_counter() - t0)
times_block = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    matcher_big.match(rules=rules, blocking_key="last_initial")
    times_block.append(time.perf_counter() - t0)
t_no_block = sum(times_no) / N_TRIALS
t_with_block = sum(times_block) / N_TRIALS
print(f"Exact — without blocking:  {t_no_block:.2f} s")
print(f"Exact — with last_initial: {t_with_block:.2f} s")
if t_with_block > 0:
    print(f"Exact speedup:    {t_no_block / t_with_block:.1f}×")

# Fuzzy: costlier per comparison → blocking helps even more
times_fuzzy_no = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    matcher_big.match_fuzzy(field="full_name", threshold=0.85)
    times_fuzzy_no.append(time.perf_counter() - t0)
times_fuzzy_block = []
for _ in range(N_TRIALS):
    t0 = time.perf_counter()
    matcher_big.match_fuzzy(field="full_name", threshold=0.85, blocking_key="last_initial")
    times_fuzzy_block.append(time.perf_counter() - t0)
t_fuzzy_no = sum(times_fuzzy_no) / N_TRIALS
t_fuzzy_block = sum(times_fuzzy_block) / N_TRIALS
print(f"Fuzzy — without blocking:  {t_fuzzy_no:.2f} s")
print(f"Fuzzy — with last_initial: {t_fuzzy_block:.2f} s")
if t_fuzzy_block > 0:
    print(f"Fuzzy speedup:   {t_fuzzy_no / t_fuzzy_block:.1f}×")

Timing on 25,000×25,000 (synthetic 50×), mean of 3 trials...
Exact — without blocking:  0.01 s
Exact — with last_initial: 0.03 s
Exact speedup:    0.4×
Fuzzy — without blocking:  5.09 s
Fuzzy — with last_initial: 0.39 s
Fuzzy speedup:   13.0×


### 4b. Different blocking key per rule

When you have **multiple rules**, you can pass `blocking_key` as a **list**: one entry per rule. The first rule uses the first key, the second rule uses the second key, and so on. Use `None` for a rule to run that rule without blocking. Example: block the **first rule (email)** by **first letter of last name** (`last_initial`) instead of zip—and the name rule by `last_initial` too. Same last-name initial keeps candidate pairs relevant for both rules.

In [8]:
# Same rules; different blocking key per rule: first rule (email) by last_initial, name rule by last_initial
rules = ["email_clean", ["first_name", "last_name"]]
results_per_key = matcher.match(rules=rules, blocking_key=["last_initial", "last_initial"])
m_per_key = results_per_key.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")
print("Blocking: both rules by last_initial:", f"recall={m_per_key['recall']:.2%}, matches={results_per_key.count}")

Blocking: both rules by last_initial: recall=66.67%, matches=20


## 5. Blocking with fuzzy

You can pass `blocking_key` to `match_fuzzy()` as well—same idea as in 04 and 05: fuzzy comparisons happen only within blocks, so the comparison count is reduced the same way. Below: fuzzy on `full_name` (as in 04) with `zip_code` blocking. Blocking applies to both exact and fuzzy matching.

In [6]:
# Same fuzzy setup as 04: full_name, threshold 0.85; with zip_code blocking
fuzzy_block = matcher.match_fuzzy(field="full_name", threshold=0.85, blocking_key="zip_code")
print(f"Fuzzy matches with zip_code blocking: {fuzzy_block.count}")

Fuzzy matches with zip_code blocking: 72


You now have **blocking** (06). Next: [07 Deduplication](07_deduplication.ipynb) — same ideas on a single table with `Deduplicator`.